# Plitt & Giocomo (2021) on dandiset 000054, part 2: population codes and inference

Paper. Plitt MH, Giocomo LM (2021). Experience-dependent contextual codes in the hippocampus.
*Nature Neuroscience* 24, 705-714.
[doi:10.1038/s41593-021-00816-6](https://doi.org/10.1038/s41593-021-00816-6). The claims file was
written from the NIH author manuscript in PubMed Central (PMC8893323) with its Methods, legends
and supplementary information.

Dandiset. [DANDI:000054](https://dandiarchive.org/dandiset/000054/0.260415.1712), version
0.260415.1712, the most recent published version; the paper cites only the draft, and the 2021
versions hold 10 of the 16 mice (see part 1, `01_sessions_place_cells.ipynb`, for the choice and
the data check of every file).

This notebook tests the population similarity, SF, Q estimate and model claims. The priors (M7)
need every trial a mouse ran before session 8; the dandiset holds only the imaged trials of the
imaging sessions. Following the answer of 2026-10-22 (questions/000054.yaml), the priors are built
from the authors' "Unprocessed VR data" on figshare
([doi:10.6084/m9.figshare.31974756.v1](https://doi.org/10.6084/m9.figshare.31974756.v1)), which holds
every VR session, read by HTTP range requests into memory, file by file. Its mice are named by
alias; the aliases of R1 to F6 are taken from the 2026 code (`src/morph_analyses/mouse_metadata.py`
at commit 9230b0a of https://github.com/GiocomoLab/PlittGiocomo_2021). The link claim
vr-deposit-link tests the match of its sessions to the dandiset's files, and
training-intermediate-fraction uses the deposit alone (`data: elsewhere`). Claims whose posteriors or priors use it are `data: with_inputs`, and
fig7-model-reproduces-remapping, which needs nothing else, is `data: elsewhere`. The earlier run's
priors from the dandiset's imaged trials are shown as a variant, to show what the answer changed.

| Claim | Status |
|---|---|
| vr-deposit-link | tested (link claim) |
| training-intermediate-fraction | tested (deposit, data elsewhere) |
| fig3ab-block-versus-gradual-similarity | tested |
| edf7-checkerboard-sessions | tested |
| fig3f-zscored-difference-at-025 | tested |
| fig4-sf-stable-along-track | tested |
| fig6a-dkl-sign | tested (priors from the deposit) |
| fig6bc-pooled-dkl | tested (priors from the deposit) |
| fig6d-prior-reconstruction | tested (priors from the deposit) |
| map-estimate-values | tested (priors from the deposit) |
| edf9-early-sessions | tested (priors from the deposit) |
| edf9k-null-models-rare | tested (priors from the deposit) |
| edf9l-null-models-frequent | tested (priors from the deposit) |
| edf10-decision-mice-like-frequent | tested (priors from the deposit) |
| fig7-model-reproduces-remapping | tested (priors from the deposit) |
| fig7g-model-prior-reconstruction | tested (priors from the deposit) |
| edf3-simulated-posterior-population | tested (priors from the deposit) |

The claims of part 1 and the two blocked claims are listed in the summary at the end.

In [ ]:
import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import morph as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = M.asset_table()
print(f"dandiset {M.DANDISET_ID} version {M.DANDISET_VERSION}: {len(assets)} NWB files")
rf = assets.group != "decision"
pop = M.process_many([r for _, r in assets[rf & (assets.kind == "testing")].iterrows()], "population")
print(f"rare and frequent sessions 8 to N: {len(pop)} streamed in {time.time() - started:.0f} s")
sfs = M.process_many([r for _, r in assets[(~rf) | (rf & (assets.kind == "training") & (assets.day == 3))].iterrows()], "sf")
print(f"decision mice (all sessions) and session 3 of rare and frequent mice: {len(sfs)}; {time.time() - started:.0f} s")
meta = M.process_many([r for _, r in assets[rf & (assets.kind == "training") & (assets.day != 3)].iterrows()], "meta", workers=8)
print(f"other training files, trials only: {len(meta)}; total {time.time() - started:.0f} s")
vr_ses, vr_trials = M.vr_deposit()
print(f"VR deposit {M.VR_DEPOSIT['doi']}: {len(vr_ses)} sessions of the 12 mice on {M.VR_TRACK}; total {time.time() - started:.0f} s")
RARE = ["R1", "R2", "R3", "R4", "R5", "R6"]
FREQ = ["F1", "F2", "F3", "F4", "F5", "F6"]
GROUP = {**{m: "rare" for m in RARE}, **{m: "frequent" for m in FREQ}}

## Data check

In [ ]:
allsess = pop + sfs + meta
chk = pd.DataFrame([{"mouse": s["mouse"], "group": s["group"], "kind": s["kind"], "day": s["day"], "part": s["part"],
                     "cells": s["n_cells"], "trials": len(s["trials"]), "frames": s["n_frames"],
                     "S0 trials": int((s["trials"].morph == 0).sum()), "S1 trials": int((s["trials"].morph == 1).sum()),
                     "starts_match": s["starts_match"], "stops_match": s["stops_match"],
                     "norm_check": s.get("norm_check"), "speed_kept": s.get("speed_kept"),
                     "empty_bins": s.get("empty_bins_a")} for s in allsess])
print("Files read, by group and kind:")
print(pd.crosstab(chk.group, chk.kind).to_string())
print("Trial tables match the flag series in every file:", bool(chk.starts_match.all() and chk.stops_match.all()))
print("Largest difference between the file's normalised series and activity / 99th percentile:",
      float(chk.norm_check.max()))
print("Fraction of frames at 2 cm/s or more, per session: "
      f"{chk.speed_kept.min():.3f} to {chk.speed_kept.max():.3f}; empty trial x bin cells after the speed "
      f"exclusion: up to {chk.empty_bins.max():.4f}")
print("\nSessions with fewer than two S-tilde = 0 or = 1 trials (no SF possible):")
print(chk[(chk["S0 trials"] < 2) | (chk["S1 trials"] < 2)][["mouse", "kind", "day", "trials", "S0 trials", "S1 trials"]].to_string(index=False))
tr = pd.concat([s["trials"].assign(mouse=s["mouse"], group=s["group"], day=s["day"]) for s in allsess])
print(f"\nS from {tr.S.min():.3f} to {tr.S.max():.3f}; f_h from {tr.fh.min():.2f} to {tr.fh.max():.2f}; "
      f"grid {M.unit_to_fh(M.GRID[0]):.2f} to {M.unit_to_fh(M.GRID[-1]):.2f} f_h, 1 unit = {M.UNIT_FH:.2f} f_h")
print("S-tilde = 0 and 1 trials map to units", M.wallmorphx(0.0).round(3), "and", M.wallmorphx(1.0).round(3))
sf_all = np.concatenate([s["sf"] for s in pop])
print(f"Whole-trial SF, sessions 8 to N: {np.nanmin(sf_all):.3f} to {np.nanmax(sf_all):.3f}, NaN {np.isnan(sf_all).sum()}")

## vr-deposit-link

> The sessions of the authors' VR deposit on figshare can be matched to the dandiset's imaging
> sessions of the rare and frequent morph mice, by mouse alias and trial sequence.

A link claim added by this attempt (protocol, Author 1.4): the priors of this notebook and
training-intermediate-fraction take the deposit's sessions of each mouse before its session 8, so
the deposit's mice and dates have to be those of the dandiset's files.

In [ ]:
nwb_rf = [s for s in allsess if s["group"] != "decision"]
named = [s for s in allsess for a in M.ALIAS.values()
         if a in " ".join([str(s["subject"]), s["session_id"], str(s["json_mouse"]), s["description"]])]
print("NWB files whose subject, session id, annotation or description name a deposit alias:", len(named))
link = M.link_sessions(nwb_rf, vr_ses, vr_trials)
print("Deposit, the 12 mice on the task track: files", len(vr_ses), "; listed twice in the session database:",
      int((vr_ses.db_rows > 1).sum()), "; listed but missing from the zip:", int((~vr_ses.in_zip).sum()),
      "; trials without a morph value:", int(vr_ses.trials_no_morph.sum()))
print(vr_ses.groupby("mouse").agg(files=("member", "size"), trials=("trials", "sum"), first=("date", "min"),
                                  last=("date", "max"), imaging_flag=("Imaging", lambda x: int((x == 1).sum()))).to_string())
print("\nEach NWB file against every deposit file (S-tilde and the three jitters within 1e-4, as one contiguous run):")
print(link[["mouse", "kind", "day", "part", "nwb_trials", "matches", "other_mouse_matches", "member", "offset",
            "vr_trials", "max_diff"]].to_string(index=False))
one = (link.matches == 1) & (link.other_mouse_matches == 0)
print(f"\nNWB files of the 12 mice: {len(link)}; matching exactly one file of their own mouse and none of another: "
      f"{int(one.sum())} ({one.mean():.0%})")
print("Matched file holds exactly the NWB trials (offset 0, same count):",
      int(((link.offset == 0) & (link.vr_trials == link.nwb_trials)).sum()), "of", len(link),
      "; largest difference of S-tilde or jitter:", float(link.max_diff.max()))
print("Matched files carry the database's imaging flag:", int((link.imaging == 1).sum()), "of", len(link))
print("Matched files are distinct:", link.member.nunique() == len(link))
d8 = link[link.day == 8].set_index("mouse")[["date", "session_number", "member"]]
print("\nSession 8 of each mouse in the deposit (the NWB file of day 8):")
print(d8.to_string())
img = vr_ses[(vr_ses.Imaging == 1) & ~vr_ses.member.isin(link.member)]
print("\nDeposit files with the imaging flag that match no NWB file:", len(img))
print(img.groupby("mouse").agg(files=("member", "size"), trials=("trials", "sum"),
                               dates=("DateFolder", lambda d: " ".join(sorted(set(d))))).to_string())

**Result: reproduced.** All 89 NWB files of the 12 rare and frequent morph mice match exactly
one deposit file of their own mouse, under the 2026 code's alias, and no file of another mouse;
in every one the deposit file holds exactly the NWB trials (offset 0, same count), with S-tilde
and the three jitters identical, and every matched file carries the database's imaging flag. So
the deposit's aliases and dates are those of the dandiset's mice and sessions, and session 8 of
each mouse has a date in the deposit. The deposit holds more than the dandiset: every session
without imaging, R6's sessions before session 8, and further files with the imaging flag that
match no NWB file (table above). No NWB file names a deposit alias, so the mapping of the
aliases to the mice comes from the 2026 code alone; the trial sequences confirm it.

## training-intermediate-fraction

> Before session 8, rare morph mice had fewer than 4% of trials with intermediate morph values,
> while frequent morph mice had 60%.

Answer of 2026-10-22: the trials come from the figshare deposit (`data: elsewhere`), which holds
every session. Trials before session 8 are those of every session on the task track on a date
before the date of session 8 (vr-deposit-link), and of the earlier session numbers of that date.
The variants do not decide: the 2026 code's two choices (sessions with more than 40 rewards; the
sessions within 8 calendar days of the mouse's first), and the imaged trials the dandiset holds,
which decided nothing in the earlier run either.

In [ ]:
def before8(m):
    return M.sessions_before(vr_ses, m, d8.loc[m, "date"], d8.loc[m, "session_number"])


def first8(m):
    s = vr_ses[vr_ses.mouse == m]
    return s[(s.date - s.date.min()).dt.days < 8]


variants = {"every trial before session 8": before8,
            "code: more than 40 rewards": lambda m: before8(m)[before8(m).RewardCount > 40],
            "code 2026: within 8 days of the first": first8}
rows = []
for vname, f in variants.items():
    for m in RARE + FREQ:
        b = f(m)
        t = pd.concat([vr_trials[x] for x in b.member]).dropna(subset=["morph"])
        rows.append({"variant": vname, "group": "rare" if m in RARE else "frequent", "mouse": m, "sessions": len(b),
                     "days": b.date.nunique(), "trials": len(t), "intermediate": int(t.morph.isin([0.25, 0.5, 0.75]).sum())})
tf = pd.DataFrame(rows)
tf["fraction"] = (tf.intermediate / tf.trials).round(4)
early = tr[(tr.day < 8) & (tr.group != "decision")]
for m in RARE + FREQ:
    t = early[early.mouse == m]
    rows.append({"variant": "dandiset imaged trials", "group": "rare" if m in RARE else "frequent", "mouse": m,
                 "sessions": t.day.nunique(), "days": t.day.nunique(), "trials": len(t),
                 "intermediate": int(((t.morph > 0.01) & (t.morph < 0.99)).sum())})
tf = pd.DataFrame(rows)
tf["fraction"] = np.where(tf.trials > 0, tf.intermediate / tf.trials.clip(lower=1), np.nan).round(4)
print("Per mouse, sessions, days, trials and fraction with S-tilde 0.25, 0.5 or 0.75 before session 8:")
print(tf.pivot(index="mouse", columns="variant", values="fraction").loc[RARE + FREQ].to_string())
print(tf[tf.variant == "every trial before session 8"][["mouse", "sessions", "days", "trials", "intermediate"]].to_string(index=False))
print("\nS-tilde values in the deposit trials before session 8:",
      sorted(set(pd.concat([vr_trials[x] for m in RARE + FREQ for x in before8(m).member]).morph.dropna())))
res = []
for vname in tf.variant.unique():
    x = tf[tf.variant == vname]
    for g in ("rare", "frequent"):
        y = x[(x.group == g) & (x.trials > 0)]
        res.append({"variant": vname, "group": g, "a: pooled": round(y.intermediate.sum() / y.trials.sum(), 4),
                    "b: lowest mouse": y.fraction.min(), "b: highest mouse": y.fraction.max()})
res = pd.DataFrame(res)
print("\n" + res.to_string(index=False))
for vname in tf.variant.unique():
    r = res[res.variant == vname].set_index("group")
    ok_a = r.loc["rare", "a: pooled"] < 0.04 and abs(r.loc["frequent", "a: pooled"] - 0.6) <= 0.06
    ok_b = r.loc["rare", "b: highest mouse"] < 0.04 and abs(r.loc["frequent", "b: lowest mouse"] - 0.6) <= 0.06 \
        and abs(r.loc["frequent", "b: highest mouse"] - 0.6) <= 0.06
    print(f"{vname}: reading (a) holds {ok_a}; reading (b) holds {ok_b}")

**Result: reproduced.** From every trial the deposit holds before session 8, 3.49% of the rare
morph mice's trials pooled (340 of 9,753) and 59.8% of the frequent morph mice's (5,700 of 9,527)
are intermediate, so reading (a) holds: below 4%, and within 0.06 of 60%. Reading (b) does not,
as R4 (4.1%) and R6 (5.7%) are above 4%; the frequent morph mice range from 59.2% to 60.6%. The
2026 code's variants give the same under reading (a) (3.41% and 3.19% rare, 59.8% and 59.8%
frequent). The imaged trials the dandiset holds give 13.8% for the rare morph mice, as the
earlier run found; the answer replaced them with the deposit.

## Priors and posteriors (M7, M8)

Priors (M7): every trial each mouse ran before session 8, from the authors' VR deposit on
figshare (answer of 2026-10-22). The deposit's sessions are matched to the dandiset's files as in
vr-deposit-link above; session 8's date and session number in
the deposit mark the end of the trials that enter the prior.

In [ ]:
vr_before8 = {m: pd.concat([vr_trials[x] for x in M.sessions_before(vr_ses, m, d8.loc[m, "date"], d8.loc[m, "session_number"]).member])
              .dropna(subset=["morph"]) for m in RARE + FREQ}
print("Deposit trials before session 8, per mouse (S-tilde counts):")
print(pd.DataFrame({m: t.morph.value_counts() for m, t in vr_before8.items()}).T.fillna(0).astype(int).to_string())
mouse_prior = {m: M.prior_from_units(t.unit.to_numpy()) for m, t in vr_before8.items()}
prior = {"rare": M.group_prior([mouse_prior[m] for m in RARE]),
         "frequent": M.group_prior([mouse_prior[m] for m in FREQ])}
# 2020 code variant: the group prior is the last mouse's prior
prior_last = {"rare": mouse_prior["R6"], "frequent": mouse_prior["F6"]}
# the earlier run's priors, from the imaged trials before session 8 that the dandiset holds (R6 has none)
before8 = tr[(tr.day < 8) & tr.group.isin(["rare", "frequent"])]
mouse_prior_img = {m: M.prior_from_units(before8[before8.mouse == m].unit.to_numpy()) for m in before8.mouse.unique()}
prior_img = {"rare": M.group_prior([mouse_prior_img[m] for m in RARE if m in mouse_prior_img]),
             "frequent": M.group_prior([mouse_prior_img[m] for m in FREQ if m in mouse_prior_img])}
POST = {(lik, g): M.posterior(prior[g], lik) for lik in M.LIKELIHOODS for g in ("rare", "frequent")}
POST_LAST = {g: M.posterior(prior_last[g]) for g in ("rare", "frequent")}
POST_IMG = {g: M.posterior(prior_img[g]) for g in ("rare", "frequent")}
mass = lambda p: {f"{lo}-{hi}": round(float(p[(M.GRID >= lo) & (M.GRID < hi)].sum()), 3) for lo, hi in ((-0.3, 0.15), (0.15, 0.85), (0.85, 1.3))}
print("Prior mass by unit range (low extreme, intermediate, high extreme):")
print("deposit (main):", {g: mass(p) for g, p in prior.items()})
print("dandiset imaged trials (earlier run):", {g: mass(p) for g, p in prior_img.items()})
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(M.unit_to_fh(M.GRID), prior["rare"], label="rare"); ax[0].plot(M.unit_to_fh(M.GRID), prior["frequent"], label="frequent")
ax[0].plot(M.unit_to_fh(M.GRID), prior_img["rare"], "--", label="rare, dandiset trials")
ax[0].set_xlabel("f_h"); ax[0].legend(); ax[0].set_title("group priors")
ax[1].imshow(POST[("methods var 8.23", "rare")], origin="lower", aspect="auto", extent=[M.GRID[0], M.GRID[-1]] * 2)
ax[1].set_title("rare posterior (rows fhat)")
plt.show()
print("prior ranges:", {g: (float(p.min()), float(p.max())) for g, p in prior.items()},
      "; dandiset-trial rare prior up to", float(prior_img["rare"].max()))

Every NWB file of the 12 mice matches one deposit file of its mouse, so session 8 has a date in
the deposit for every mouse, R6 included (its training sessions are in the deposit, not the
dandiset). Over all trials before session 8, 3.5% of the rare morph mice's trials are
intermediate (part 1), and the rare prior holds 0.183 of its mass between 0.15 and 0.85 units,
against 0.256 for the prior from the dandiset's imaged trials that the earlier run used. The
frequent prior hardly changes (0.411 against 0.419). The earlier run's priors are kept as a variant
labelled "earlier run", to show what the answer changed; they do not decide.

In [ ]:
# Per-session Q estimates and Delta D_KL (M9 to M11), for the readings the claims use.
def session_q(sf, trials, anchors=M.MAP_ANCHORS_FH):
    t = M.transform_sf(sf, trials["morph"].to_numpy(), anchors)
    return M.q_unnorm(t, trials["unit"].to_numpy())


def can_sf(s):
    return (s["trials"].morph == 0).sum() >= 2 and (s["trials"].morph == 1).sum() >= 2


for s in pop + [x for x in sfs if can_sf(x)]:
    s["qu"] = session_q(s["sf"], s["trials"])
    s["q"] = M.normalise_rows(s["qu"])
    s["dkl"] = {lik: M.delta_dkl(s["q"], POST[(lik, "rare")], POST[(lik, "frequent")]) for lik in M.LIKELIHOODS}
    s["dkl_last"] = M.delta_dkl(s["q"], POST_LAST["rare"], POST_LAST["frequent"])
    s["dkl_img"] = M.delta_dkl(s["q"], POST_IMG["rare"], POST_IMG["frequent"])
    if "sf_norm" in s:
        s["q_norm"] = M.normalise_rows(session_q(s["sf_norm"], s["trials"]))
        s["dkl_norm"] = M.delta_dkl(s["q_norm"], POST[("methods var 8.23", "rare")], POST[("methods var 8.23", "frequent")])
bad = sum(M.dkl_rows(POST[("methods var 8.23", g)][M.ROWS], s["q"][M.ROWS])[1] for s in pop for g in ("rare", "frequent"))
print("grid points left out for Q = 0 where P > 0:", bad)

## fig3ab-block-versus-gradual-similarity

> Single-session population trial-by-trial similarity matrices show two blocks in the rare morph
> condition and a gradual transition between the extremes in the frequent morph condition.

In [ ]:
Bm = {m: np.nanmean([s["C_raw"] for s in pop if s["mouse"] == m], axis=0) for m in RARE + FREQ}
share = {m: M.step_share(B) for m, B in Bm.items()}
for g, mice in (("rare", RARE), ("frequent", FREQ)):
    print(g, "step share per mouse:", {m: round(share[m], 3) for m in mice})
    print(f"  mean binned similarity, rows S-tilde = 0 and = 1:\n", np.round(np.mean([Bm[m] for m in mice], axis=0)[[0, 4]], 3))
obs, perms = M.mouse_permutation(share, RARE, FREQ)
print(f"rare mean {np.mean([share[m] for m in RARE]):.3f}, frequent {np.mean([share[m] for m in FREQ]):.3f}; "
      f"difference {obs:.3f}; 2.5 to 97.5 percentiles {np.percentile(perms, 2.5):.3f} to {np.percentile(perms, 97.5):.3f}; "
      f"outside: {bool(M.outside_95(obs, perms))}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, (m, d) in zip(ax, (("R2", 13), ("F4", 10))):
    s = [x for x in pop if x["mouse"] == m and x["day"] == d][0]
    a.imshow(s["C_full_sorted"], cmap="magma"); a.set_title(f"{m} session {d}, trials sorted by S")
plt.show()
print("similarity range in the two panels:", [(float(np.min(x["C_full_sorted"])), float(np.max(x["C_full_sorted"])))
                                              for x in pop if (x["mouse"], x["day"]) in (("R2", 13), ("F4", 10))])

**Result: not reproduced.** The step share (the largest fall between neighbouring S-tilde values
over the whole fall, mean of the S-tilde = 0 and = 1 rows of each mouse's binned matrix) is 0.605
for rare morph mice (0.485 to 0.792) and 0.563 for frequent morph mice (0.434 to 0.749); the
difference, 0.042, lies inside the 2.5 to 97.5 percentiles of the mouse permutations (-0.116 to
0.116). The rare morph rows fall mostly between S-tilde 0.25 and 0.5 (0.439 to 0.339) and the
frequent morph rows fall more evenly, but the measure the criterion names does not separate the
groups.

## edf7-checkerboard-sessions

> In six named rare morph sessions, trials near S-tilde = 0.25 randomly take one extreme
> representation or the other, seen as checkerboard bands in the sorted similarity matrices.

In [ ]:
named = [("R1", 8), ("R2", 13), ("R3", 9), ("R4", 11), ("R5", 8), ("R6", 10)]
ok_all = True
for m, d in named:
    s = [x for x in pop if x["mouse"] == m and x["day"] == d][0]
    sf = s["sf"][s["trials"].morph == 0.25]
    ok = (sf > 0.5).any() and (sf < 0.5).any()
    ok_all &= ok
    print(f"{m} session {d}: {len(sf)} S-tilde = 0.25 trials, SF above 0.5: {(sf > 0.5).sum()}, below: {(sf < 0.5).sum()}, "
          f"range {sf.min():.3f} to {sf.max():.3f}")
print("criterion met in all six:", ok_all)
others = [x for x in pop if x["group"] == "rare" and (x["mouse"], x["day"]) not in named]
print("other rare sessions with S-tilde = 0.25 SF on both sides of 0.5:",
      sum(((x["sf"][x["trials"].morph == 0.25] > 0.5).any() and (x["sf"][x["trials"].morph == 0.25] < 0.5).any()) for x in others),
      "of", len(others))

**Result: partially reproduced.** In five of the six named sessions the S-tilde = 0.25 trials
include SF values on both sides of 0.5 (R1 session 8: 5 above and 15 below; R2 session 13: 3 and
17; R3 session 9: 1 and 19; R4 session 11: 1 and 15; R6 session 10: 4 and 5). In R5 session 8
all 16 lie below 0.5 (0.424 to 0.465). Most other rare sessions (18 of 20) also have S-tilde =
0.25 trials on both sides.

## fig3f-zscored-difference-at-025

> After z-scoring within session, trial-by-trial population similarity differs significantly
> between rare and frequent morph mice for S-tilde = 0.25 trials.

In [ ]:
def zdiff(key, mouse_z=False, rare=RARE, freq=FREQ):
    if mouse_z:
        per = {m: M.zscore_matrix(np.nanmean([s[key] for s in pop if s["mouse"] == m], axis=0)) for m in rare + freq}
    else:
        per = {m: np.nanmean([M.zscore_matrix(s[key]) for s in pop if s["mouse"] == m], axis=0) for m in rare + freq}
    return M.mouse_permutation(per, rare, freq)


fig3f = {}
for label, key, mz in (("main: smoothed deconvolved, z-scored per session", "C_raw", False),
                       ("variant: 99th-percentile normalised signal", "C_norm", False),
                       ("variant: unsmoothed (2020 code)", "C_unsmoothed", False),
                       ("variant: z-scored mouse means (2026 code)", "C_raw", True)):
    obs, perms = zdiff(key, mz)
    sig = M.outside_95(obs, perms)
    fig3f[label] = (obs, perms, sig)
    print(label)
    print(pd.DataFrame(obs, index=M.MORPHS, columns=M.MORPHS).round(3).to_string())
    lo, hi = np.percentile(perms, [2.5, 97.5], axis=0)
    print("  S-tilde = 0.25 row: difference", obs[1].round(3), "\n  2.5 pct", lo[1].round(3), "\n  97.5 pct", hi[1].round(3))
    print("  significant entries of the 0.25 row:", [(M.MORPHS[j], round(obs[1, j], 3)) for j in range(5) if sig[1, j]])
    print("  all significant entries:", [(M.MORPHS[i], M.MORPHS[j], round(obs[i, j], 3)) for i, j in zip(*np.nonzero(sig)) if i <= j])

**Result: reproduced.** In the S-tilde = 0.25 row of the z-scored rare minus frequent
difference, the entries with S-tilde = 0.5 (-0.409) and 1.0 (-0.367) lie outside the 2.5 to 97.5
percentiles of the 923 mouse permutations: rare morph 0.25 trials are less similar to 0.5 and 1.0
trials than frequent morph ones. The entry with S-tilde = 0 is +0.281 against a 97.5 percentile of
0.290. The variants (normalised signal, no smoothing, z-scoring the mouse means) also find
significant entries in the 0.25 row.

## fig4-sf-stable-along-track

> Position-wise SF shows stable clustered representations along the whole track in rare morph
> mice and a spectrum in frequent morph mice; remapping happens soon after the trial starts and
> stays as the mouse runs.

In [ ]:
curves = {}
for m in RARE + FREQ:
    per_sess = []
    for s in pop:
        if s["mouse"] != m:
            continue
        mor = s["trials"].morph.to_numpy()
        per_sess.append(np.array([np.nanmean(s["sf_pos"][mor == v], axis=0) for v in M.MORPHS]))
    curves[m] = np.nanmean(per_sess, axis=0)            # 5 x bins
res4 = {}
for g, mice in (("rare", RARE), ("frequent", FREQ)):
    c = np.array([curves[m] for m in mice])
    mean, sem = c.mean(0), c.std(0, ddof=1) / np.sqrt(len(mice))
    res4[g] = (mean, sem)
    sep = np.abs(mean[0] - mean[4]) > sem[0] + sem[4]
    print(f"{g}: S-tilde 0 and 1 separated by more than the summed s.e.m. in {sep[1:].sum()} of 44 bins after the first")
    print("  mean SF by S-tilde, every 5th bin:\n", np.round(mean[:, ::5], 3))
mean, sem = res4["rare"]
ov = np.abs(mean[3] - mean[4]) <= sem[3] + sem[4]
near0 = np.abs(mean[1] - mean[0]) < np.abs(mean[1] - mean[4])
print(f"rare: 0.75 and 1.0 overlap within s.e.m. in {ov.sum()} of 45 bins; 0.25 nearer 0 than 1 in {near0.sum()} of 45 "
      f"bins ({near0[1:].sum()} of 44 after the first)")
mean, sem = res4["frequent"]
between = (np.minimum(mean[0], mean[2]) < mean[1]) & (mean[1] < np.maximum(mean[0], mean[2]))
print(f"frequent: 0.25 between 0 and 0.5 in {between.sum()} of 45 bins")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for a, g in zip(ax, ("rare", "frequent")):
    mean, sem = res4[g]
    for k, v in enumerate(M.MORPHS):
        a.plot(np.arange(45) * 10 + 5, mean[k], color=plt.cm.cool(v), label=str(v))
        a.fill_between(np.arange(45) * 10 + 5, mean[k] - sem[k], mean[k] + sem[k], color=plt.cm.cool(v), alpha=0.3)
    a.set_title(g); a.set_xlabel("position (cm)")
ax[0].set_ylabel("SF"); ax[0].legend()
plt.show()
print("plotted range:", {g: (float((res4[g][0] - res4[g][1]).min()), float((res4[g][0] + res4[g][1]).max())) for g in res4})

**Result: reproduced.** In both conditions the S-tilde = 0 and = 1 curves of position-wise SF are
separated by more than their summed s.e.m. in all 44 bins after the first. In rare morph mice the
S-tilde 0.75 and 1.0 curves overlap within s.e.m. in all 45 bins and the 0.25 curve lies nearer the
S-tilde 0 curve in all 45. In frequent morph mice the 0.25 curve lies between the 0 and 0.5 curves
in all 45 bins. The curves are flat along the track (printed every 5th bin).

## fig6a-dkl-sign

> Delta D_KL of each session's Q estimate took the expected sign (negative for rare, positive for
> frequent) in 41 of 45 sessions and in every session of 9 of 12 mice; across mice, rare -0.88 +-
> 0.23 and frequent 0.50 +- 0.11 (mean +- s.e.m.), a significant difference.

In [ ]:
def fig6a_summary(values, label):
    d = pd.DataFrame([{"mouse": s["mouse"], "group": s["group"], "day": s["day"], "dkl": v} for s, v in zip(pop, values)])
    d["expected"] = np.where(d.group == "rare", d.dkl < 0, d.dkl > 0)
    per = d.groupby("mouse").dkl.mean()
    every = d.groupby("mouse").expected.all()
    obs, perms = M.mouse_permutation(per.to_dict(), RARE, FREQ)
    r, f = per[RARE], per[FREQ]
    out = {"reading": label, "sessions expected sign": f"{int(d.expected.sum())} of {len(d)}",
           "mice every session": int(every.sum()), "rare mean": round(r.mean(), 3), "rare sem": round(r.sem(), 3),
           "frequent mean": round(f.mean(), 3), "frequent sem": round(f.sem(), 3), "difference": round(obs, 3),
           "perm range": (round(perms.min(), 3), round(perms.max(), 3)), "outside range": bool(M.outside_range(obs, perms))}
    return out, d


rows = []
main6a, d6a = fig6a_summary([s["dkl"]["methods var 8.23"] for s in pop], "main: Methods likelihood")
rows.append(main6a)
for lik in ("code 0.3 units", "code 0.4 units"):
    rows.append(fig6a_summary([s["dkl"][lik] for s in pop], f"variant: likelihood {lik}")[0])
rows.append(fig6a_summary([s["dkl_last"] for s in pop], "variant: last mouse's prior (2020 code)")[0])
rows.append(fig6a_summary([s["dkl_norm"] for s in pop], "variant: normalised signal")[0])
rows.append(fig6a_summary([s["dkl_img"] for s in pop], "earlier run: priors from dandiset trials")[0])
print(pd.DataFrame(rows).to_string(index=False))
print("\nExpected: 41 of 45 sessions (+-4.1), 9 mice (+-0.9), rare -0.88 +- 0.23, frequent 0.50 +- 0.11")
print("\nPer session, main reading:")
print(d6a.pivot_table(index="mouse", columns="day", values="dkl").round(2).to_string())
m = d6a.groupby("mouse").dkl.mean()
print("per mouse:", m.round(3).to_dict())
print(f"\nwithin tolerance: sessions {abs(int(d6a.expected.sum()) - 41) <= 4.1}, mice {abs(main6a['mice every session'] - 9) <= 0.9}, "
      f"rare {abs(main6a['rare mean'] + 0.88) <= 0.23}, frequent {abs(main6a['frequent mean'] - 0.50) <= 0.11}")

**Result: partially reproduced.** With the deposit's priors and the likelihood of the Methods,
Delta D_KL has the expected sign in 43 of 47 sessions (41 +- 4.1 expected) and the difference
between groups, -0.817, lies outside the range of the 923 permutations (-0.727 to 0.817; the
largest permutation is the swapped grouping). The rare mean, -0.660 +- 0.104, is within tolerance
(-0.88 +- 0.23); the frequent mean, 0.158 +- 0.052, has the expected sign but is below 0.50 +- 0.11,
and 8 mice have the expected sign in every session (9 +- 0.9 expected). The test and the signs
hold and two values are outside tolerance. The priors from the dandiset's imaged trials, used in
the earlier run, gave 37 of 47 sessions, 7 mice, rare -0.354 and frequent 0.007. With the code's
likelihood widths, variants, 40 to 44 sessions, 10 mice, rare -0.848 and -0.866, frequent 0.701
and 0.778.

## fig6bc-pooled-dkl

> Q estimates pooled over all sessions and mice approximate the posterior of the matching
> condition (Delta D_KL rare -0.44, frequent 0.42).

In [ ]:
def group_q(mice, reading, sess=pop):
    qs = []
    for m in mice:
        ss = [s for s in sess if s["mouse"] == m and "q" in s]
        if not ss:
            continue
        if reading == "a":
            qm = M.normalise_rows(np.mean([s["q"] for s in ss], axis=0))
        else:
            qm = M.normalise_rows(np.sum([s["qu"] for s in ss], axis=0))
        qs.append(qm)
    return M.normalise_rows(np.sum(qs, axis=0)), len(qs)


for reading in "ab":
    out = {}
    for g, mice in (("rare", RARE), ("frequent", FREQ)):
        q, n = group_q(mice, reading)
        out[g] = {lik: round(M.delta_dkl(q, POST[(lik, "rare")], POST[(lik, "frequent")]), 3) for lik in M.LIKELIHOODS}
    print(f"reading {reading} ({'mean of session Q' if reading == 'a' else 'one Q from all trials of a mouse'}):", out)
out = {g: round(M.delta_dkl(group_q(mice, "a")[0], POST_IMG["rare"], POST_IMG["frequent"]), 3)
       for g, mice in (("rare", RARE), ("frequent", FREQ))}
print("earlier run, priors from dandiset trials, reading a:", out)
print("expected rare -0.44 +- 0.044, frequent 0.42 +- 0.042 (main likelihood decides)")

**Result: partially reproduced.** With the deposit's priors, pooling the Q estimates gives rare
-0.464 and frequent 0.125 when each mouse's Q is the mean of its session Q estimates (reading a),
and -0.469 and 0.118 when each mouse's Q comes from all its trials (reading b), against -0.44 +-
0.044 and 0.42 +- 0.042. The rare value is within tolerance under both readings; the frequent
value has the expected sign but is far below 0.42, so only the signs hold. The earlier run's
priors from the dandiset's imaged trials gave rare -0.266 and frequent -0.000. With the code's
likelihood widths, variants, the frequent value is 0.590 to 0.646 and the rare -0.259 to -0.360.

## fig6d-prior-reconstruction

> Priors reconstructed from each mouse's neural data alone are closer to the ideal prior of the
> mouse's own condition (Delta D_KL rare -0.46 +- 0.12, frequent 0.46 +- 0.07, mean +- s.e.m.
> across mice), a significant group difference.

In [ ]:
def prior_dkl(qu, lik="methods var 8.23", priors=prior):
    R = M.reconstruct_prior(qu, lik)
    return M.dkl(priors["rare"], R) - M.dkl(priors["frequent"], R), R


recon = {}
rows = []
for lik in M.LIKELIHOODS:
    vals = {}
    for m in RARE + FREQ:
        qu = np.sum([s["qu"] for s in pop if s["mouse"] == m], axis=0)
        vals[m], R = prior_dkl(qu, lik)
        if lik == "methods var 8.23":
            recon[m] = R
    obs, perms = M.mouse_permutation(vals, RARE, FREQ)
    r, f = pd.Series({m: vals[m] for m in RARE}), pd.Series({m: vals[m] for m in FREQ})
    rows.append({"likelihood": lik, "rare mean": round(r.mean(), 3), "rare sem": round(r.sem(), 3),
                 "frequent mean": round(f.mean(), 3), "frequent sem": round(f.sem(), 3), "difference": round(obs, 3),
                 "outside range": bool(M.outside_range(obs, perms)), "rare signs negative": int((r < 0).sum()),
                 "frequent signs positive": int((f > 0).sum())})
    if lik == "methods var 8.23":
        fig6d_vals = vals
        print("per mouse (main):", {k: round(v, 3) for k, v in vals.items()})
print(pd.DataFrame(rows).to_string(index=False))
vals_img = {m: prior_dkl(np.sum([s["qu"] for s in pop if s["mouse"] == m], axis=0), priors=prior_img)[0] for m in RARE + FREQ}
print("earlier run, priors from dandiset trials: rare mean", round(np.mean([vals_img[m] for m in RARE]), 3),
      "frequent mean", round(np.mean([vals_img[m] for m in FREQ]), 3))
print("expected rare -0.46 +- 0.12, frequent 0.46 +- 0.07")
fig, ax = plt.subplots(figsize=(6, 3))
for m in RARE + FREQ:
    ax.plot(M.unit_to_fh(M.GRID), recon[m], color="tab:blue" if m in RARE else "tab:orange", lw=0.8)
ax.plot(M.unit_to_fh(M.GRID), prior["rare"], "k--"); ax.plot(M.unit_to_fh(M.GRID), prior["frequent"], "k:")
ax.set_xlabel("f_h"); plt.show()
print("reconstructed prior range:", float(min(r.min() for r in recon.values())), float(max(r.max() for r in recon.values())))

**Result: not reproduced.** The reconstructed priors still separate the groups: the difference of
across-mouse means, -0.902, lies outside the range of all permutations, and all six frequent morph
mice are positive (1.125 +- 0.076, expected 0.46 +- 0.07). But with the deposit's priors the rare
morph mean is positive, 0.223 +- 0.122 (expected -0.46 +- 0.12), with only R3 (-0.033) and R6
(-0.166) negative, so the sign of the rare mean fails and the criterion's partial outcome (the
test and the signs) is not met either: four of the six rare morph mice's reconstructions are
nearer the frequent prior than the rare prior.
The earlier run's priors from the dandiset's imaged trials gave rare -0.020 and frequent 0.633;
the code's likelihood widths, variants, give rare 0.480 and 0.517, frequent 1.285 and 1.379.

## map-estimate-values

> The average MAP estimate of f_h for S-tilde = 0 trials is 11.08 under the rare and 10.64 under
> the frequent posterior, and for S-tilde = 1 trials 38.13 and 37.30.

In [ ]:
exp_map = {("rare", 0): 11.08, ("frequent", 0): 10.64, ("rare", 1): 38.13, ("frequent", 1): 37.30}
ttr = pd.concat([s["trials"].assign(group=s["group"]) for s in pop])
for lik in M.LIKELIHOODS:
    for reading in ("a: all sessions", "b: own group's sessions"):
        vals = {}
        for (g, v), e in exp_map.items():
            x = ttr[ttr.morph == v] if reading.startswith("a") else ttr[(ttr.morph == v) & (ttr.group == g)]
            vals[(g, v)] = float(np.mean(M.unit_to_fh(M.map_estimate(POST[(lik, g)], x.unit.to_numpy()))))
        ok = all(abs(vals[k] - e) <= 0.1 * e for k, e in exp_map.items())
        print(f"{lik:18s} {reading}: " + ", ".join(f"{g} S{v} {vals[(g, v)]:.2f}" for (g, v) in exp_map) + f"; all within 10%: {ok}")
vals_last = {(g, v): float(np.mean(M.unit_to_fh(M.map_estimate(POST_LAST[g], ttr[ttr.morph == v].unit.to_numpy())))) for (g, v) in exp_map}
vals_img = {(g, v): float(np.mean(M.unit_to_fh(M.map_estimate(POST_IMG[g], ttr[ttr.morph == v].unit.to_numpy())))) for (g, v) in exp_map}
print("earlier run, priors from dandiset trials, reading a:", {f"{g} S{v}": round(x, 2) for (g, v), x in vals_img.items()})
print("variant last-mouse prior, reading a:", {f"{g} S{v}": round(x, 2) for (g, v), x in vals_last.items()})
print("f_h of the S-tilde = 0 and 1 trials themselves: means",
      round(float(ttr[ttr.morph == 0].fh.mean()), 2), round(float(ttr[ttr.morph == 1].fh.mean()), 2))

**Result: reproduced.** With the deposit's priors and the Methods likelihood the mean MAP
estimates over all S-tilde = 0 and = 1 trials of sessions 8 to N are 10.62 and 10.69 (rare and
frequent posterior) and 38.19 and 37.62, all within 10% of 11.08, 10.64, 38.13 and 37.30; each
group's own trials (reading b) and the code's likelihood widths give values within 10% as well.
The earlier run's priors gave 10.67, 10.72, 38.09 and 37.62.

## edf9-early-sessions

> The rare versus frequent remapping patterns were stable over sessions and emerged early in
> training (session 3) in most animals.

In [ ]:
s3 = [s for s in sfs if s["day"] == 3 and s["group"] in ("rare", "frequent") and "q" in s]
# reading (b): every deposit trial before session 3 (the date and session number of the day 3 file)
d3 = link[link.day == 3].set_index("mouse")
mp3, n3 = {}, {}
for m in d3.index:
    b3 = M.sessions_before(vr_ses, m, d3.loc[m, "date"], d3.loc[m, "session_number"])
    t3 = pd.concat([vr_trials[x] for x in b3.member]).dropna(subset=["morph"])
    mp3[m], n3[m] = M.prior_from_units(t3.unit.to_numpy()), (len(b3), len(t3), round(float(t3.morph.isin([0.25, 0.5, 0.75]).mean()), 3))
prior3 = {"rare": M.group_prior([mp3[m] for m in RARE if m in mp3]), "frequent": M.group_prior([mp3[m] for m in FREQ if m in mp3])}
post3 = {g: M.posterior(prior3[g]) for g in prior3}
print("deposit sessions, trials and intermediate fraction before session 3, per mouse:", n3)
rows = []
for s in s3:
    a = s["dkl"]["methods var 8.23"]
    b = M.delta_dkl(s["q"], post3["rare"], post3["frequent"])
    exp_sign = (lambda v: v < 0) if s["group"] == "rare" else (lambda v: v > 0)
    rows.append({"mouse": s["mouse"], "group": s["group"], "dkl a": round(a, 3), "dkl b": round(b, 3),
                 "expected a": exp_sign(a), "expected b": exp_sign(b)})
r9 = pd.DataFrame(rows)
print(r9.to_string(index=False))
print(f"session 3 with the expected sign: reading a {int(r9['expected a'].sum())} of {len(r9)}, reading b {int(r9['expected b'].sum())} of {len(r9)}")
cons = d6a.groupby("mouse").apply(lambda d: max((d.dkl < 0).mean(), (d.dkl > 0).mean()))
print("sessions 8 to N, share of each mouse's sessions with its majority sign:", cons.round(2).to_dict())
print("every mouse has one sign in most of its sessions:", bool((cons > 0.5).all()))

**Result: reproduced.** Session 3 Delta D_KL has the expected sign in 8 of 11 mice with the group
posteriors of sessions 8 to N built from the deposit (reading a), and in 8 of 11 with posteriors
from every deposit trial before session 3 (reading b; all 11 mice with a session 3 have such
trials in the deposit, 320 to 517 each, none intermediate for the rare morph mice). The same three
mice (F2, R1, R5) have the other sign under both readings. In sessions 8 to N every mouse's values
have one sign in most of its sessions (67% to 100%).

## edf9k-null-models-rare and edf9l-null-models-frequent

> For rare (frequent) morph sessions, Q estimates are fit worse by the other condition's, uniform,
> Gaussian and multimodal prior posteriors and by a uniform posterior than by the matching
> posterior.

In [ ]:
unif = np.ones_like(M.GRID) / len(M.GRID)
gauss = np.exp(-((M.GRID - 0.5) / 0.2) ** 2); gauss /= gauss.sum()
multi = sum(w * np.exp(-((M.GRID - M.wallmorphx(sv)) / 0.08) ** 2) for w, sv in zip((1, 1, 10, 1, 1), M.MORPHS)); multi /= multi.sum()
nulls = {"uniform prior": M.posterior(unif), "Gaussian prior": M.posterior(gauss), "multimodal prior": M.posterior(multi),
         "uniform posterior": np.full((len(M.GRID), len(M.GRID)), 1 / len(M.GRID))}
for g, other in (("rare", "frequent"), ("frequent", "rare")):
    ss = [s for s in pop if s["group"] == g]
    own = np.array([M.mean_dkl(s["q"], POST[("methods var 8.23", g)]) for s in ss])
    print(f"=== {g} sessions ({len(ss)}); mean D_KL to own posterior {own.mean():.3f}")
    allok = True
    for name, P in {f"{other} posterior": POST[("methods var 8.23", other)], **nulls}.items():
        d = np.array([M.mean_dkl(s["q"], P) for s in ss])
        w = stats.wilcoxon(d - own)
        ok = w.pvalue < 0.05 and np.median(d - own) > 0
        allok &= ok
        print(f"  {name:20s}: mean D_KL {d.mean():.3f}, median null minus own {np.median(d - own):.3f}, "
              f"sessions with own closer {int((d > own).sum())} of {len(ss)}, Wilcoxon p = {w.pvalue:.2g}, holds {ok}")
    print(f"  all five hold: {allok}")

**Result: edf9k reproduced; edf9l partially reproduced.** With the deposit's priors, for the 26
rare morph sessions every null (frequent posterior, uniform, Gaussian and multimodal priors,
uniform posterior) fits worse than the rare posterior in all 26 sessions (Wilcoxon p = 3e-08 for
each). For the 21 frequent morph sessions four nulls fit worse than the frequent posterior: the
rare posterior (17 of 21 sessions, p = 0.00051), the Gaussian prior (p = 9.5e-07), the
multimodal prior (p = 1.3e-05) and the uniform posterior (p = 1.9e-06). The uniform prior does
not (the frequent posterior is closer in 5 of 21 sessions, p = 0.12). Four of the five frequent
morph tests hold; with the earlier run's priors three did, and the rare posterior was not worse.

## edf10-decision-mice-like-frequent

> Frequent morph mice trained to report whether S was above or below 0.5 by where they licked
> (FD1 to FD4) remapped like frequent morph mice without the task.

In [ ]:
fdm = ["FD1", "FD2", "FD3", "FD4"]
fds = [s for s in sfs if s["group"] == "decision"]
print("decision sessions with an SF (S-tilde 0 and 1 trials present):", sum("q" in s for s in fds), "of", len(fds),
      "; without:", [(s["mouse"], s["day"]) for s in fds if "q" not in s])
for reading, sel in (("a: every imaged session", lambda s: True), ("b: sessions 8 to N", lambda s: s["day"] >= 8)):
    ss = [s for s in fds if "q" in s and sel(s)]
    q, n = group_q(fdm, "a", ss)
    pooled = {lik: round(M.delta_dkl(q, POST[(lik, "rare")], POST[(lik, "frequent")]), 3) for lik in M.LIKELIHOODS}
    recs = {}
    for m in fdm:
        mm = [s for s in ss if s["mouse"] == m]
        if mm:
            recs[m] = round(prior_dkl(np.sum([s["qu"] for s in mm], axis=0))[0], 3)
    per_sess = [round(s["dkl"]["methods var 8.23"], 2) for s in ss]
    print(f"{reading}: {n} mice, {len(ss)} sessions; pooled Q Delta D_KL {pooled}; reconstructed prior Delta D_KL per mouse {recs}")
    print(f"   per-session Delta D_KL (main): {per_sess}; positive in {sum(v > 0 for v in per_sess)} of {len(per_sess)}")

**Result: reproduced.** With the deposit's priors both readings hold. (a) The Q estimate pooled
over the decision mice has Delta D_KL 0.042 over every imaged session and 0.125 over sessions 8
to N, so it is closer to the frequent posterior (with the earlier run's priors these were -0.042
and -0.006); the code's likelihood widths, variants, give 0.569 to 0.820. (b) Each decision
mouse's reconstructed prior is closer to the frequent morph prior (0.933 to 1.082 over every
imaged session, 0.970 to 1.128 over sessions 8 to N). The per-session values are positive in 25
of 35 sessions. FD1's sessions 1 and 3 have no S-tilde = 0 or 1 trials and give no SF, and FD3 has
no session after 7.

## fig7-model-reproduces-remapping and fig7g-model-prior-reconstruction

> A Hebbian network with radial basis inputs and k-winners-take-all competition, trained on
> stimuli drawn from the rare or frequent morph prior, gives Q estimates that match the matching
> posterior.
>
> Priors reconstructed from the model's Q estimates are closer to the prior of the matching
> condition, though less closely matched than for the neural data.

Answer of 2026-10-22: the model runs in the code's unit (1 unit = 27.42 f_h), with input width
sigma = 0.15 units (4.11 f_h, so a denominator of 0.0225 units^2), input centres from -0.1 to 1.1
units, stimulus, output and weight noise as in the code, in training and testing alike, and
testing on the f_h of 1,000 evenly spaced S from -0.3 to 1.3. The earlier run's readings (no
weight noise, no test noise, centres over the whole grid, the width read in f_h) are dropped. The
priors are those of M7 from the deposit. fig7-model-reproduces-remapping is `data: elsewhere`, as
it needs only the priors; fig7g-model-prior-reconstruction is `data: with_inputs`.

In [ ]:
THETA_IN = np.linspace(-0.1, 1.1, 100)          # input centres, units (code THETA)
S_TEST = np.linspace(-0.3, 1.3, 1000)            # test morph values (code)
U_TEST = M.wallmorphx(S_TEST)                     # their stimulus values, units
KW = dict(N=100, K=20, c_y=0.01, eta=0.1, tau=0.001, w_max=5.0, sigma=0.15, stim_noise=0.1, act_noise=0.01, dw_noise=0.05)


def kwta_respond(u, W, rng, p=KW):
    """Inputs and outputs for stimuli u (units): Gaussian tuning exp(-(mu - x)^2 / sigma^2) of the
    stimulus plus noise; z = W x plus 0.01 N(0,1); the K largest z give c_y z; every other output
    0.01 times the (K+1)-th largest z times N(0,1); all rectified at 0."""
    u = np.atleast_1d(u)
    s = u + p["stim_noise"] * rng.standard_normal(u.shape)
    x = np.exp(-((THETA_IN[None, :] - s[:, None]) ** 2) / p["sigma"] ** 2)
    z = x @ W.T + p["act_noise"] * rng.standard_normal((len(u), p["N"]))
    order = np.argsort(-z, axis=1)
    zk1 = np.take_along_axis(z, order[:, p["K"]:p["K"] + 1], axis=1)
    y = p["act_noise"] * zk1 * rng.standard_normal(z.shape)
    win = np.zeros_like(z, dtype=bool)
    np.put_along_axis(win, order[:, :p["K"]], True, axis=1)
    y[win] = p["c_y"] * z[win]
    return x, np.maximum(y, 0)


def hebbian_model(prior_g, rng, p=KW):
    """Methods, Hebbian learning model, with the code's scale and noise (criterion of
    fig7-model-reproduces-remapping). Returns the unnormalised Q of the test stimuli, the number of
    test stimuli without an SF, and the mean and largest weight after training."""
    W = rng.uniform(0, 0.1, (p["N"], len(THETA_IN)))       # outputs x inputs
    for u in rng.choice(M.GRID, size=1000, p=prior_g):
        x, y = kwta_respond(u, W, rng, p)
        dW = p["eta"] * y.T @ x + p["dw_noise"] * rng.standard_normal(W.shape) - p["tau"]
        W = np.clip(W + dW, 0, p["w_max"])
    _, Y = kwta_respond(U_TEST, W, rng, p)
    c0, c1 = Y[U_TEST < 0.05].mean(0), Y[U_TEST > 0.9].mean(0)
    n = np.linalg.norm(Y, axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        g0 = Y @ c0 / (n * np.linalg.norm(c0)); g1 = Y @ c1 / (n * np.linalg.norm(c1))
        sf = g1 / (g0 + g1)
    m0 = np.nanmedian(sf[(S_TEST >= -0.1) & (S_TEST <= 0.1)]); m1 = np.nanmedian(sf[(S_TEST >= 0.9) & (S_TEST <= 1.1)])
    t = np.clip((sf - m0) / (m1 - m0) * (1.073 - 0.094) + 0.094, M.GRID[0], M.GRID[-1])
    return M.q_unnorm(t, U_TEST.copy()), int(np.isnan(sf).sum()), float(W.mean()), float(W.max())


print("test stimuli span", round(float(U_TEST.min()), 3), "to", round(float(U_TEST.max()), 3), "units (",
      round(float(M.unit_to_fh(U_TEST.min())), 2), "to", round(float(M.unit_to_fh(U_TEST.max())), 2), "f_h ); input centres",
      round(float(M.unit_to_fh(THETA_IN[0])), 2), "to", round(float(M.unit_to_fh(THETA_IN[-1])), 2), "f_h; width",
      round(KW["sigma"] * M.UNIT_FH, 2), "f_h")
t0 = time.time()
model = {}
for pname, pri in (("deposit priors (main)", prior), ("earlier run: priors from dandiset trials", prior_img)):
    for gi, g in enumerate(("rare", "frequent")):
        rng = np.random.default_rng(7100 + 10 * (pname != "deposit priors (main)") + gi)
        model[(pname, g)] = [hebbian_model(pri[g], rng) for _ in range(50)]
rows = []
for (pname, g), runs in model.items():
    for lik in M.LIKELIHOODS:
        d = [M.delta_dkl(M.normalise_rows(r[0]), POST[(lik, "rare")], POST[(lik, "frequent")]) for r in runs]
        rows.append({"priors": pname, "likelihood": lik, "trained on": g, "median Delta D_KL": round(float(np.median(d)), 3),
                     "IQR": (round(float(np.percentile(d, 25)), 3), round(float(np.percentile(d, 75)), 3)),
                     "models below 0": int(np.sum(np.array(d) < 0)),
                     "models with NaN SF points": sum(r[1] > 0 for r in runs),
                     "mean weight": round(float(np.mean([r[2] for r in runs])), 3),
                     "largest weight": round(float(np.max([r[3] for r in runs])), 3)})
fig7 = pd.DataFrame(rows)
print(fig7.to_string(index=False))
for pname in fig7.priors.unique():
    for lik in M.LIKELIHOODS:
        x = fig7[(fig7.priors == pname) & (fig7.likelihood == lik)].set_index("trained on")["median Delta D_KL"]
        print(f"{pname}, {lik}: rare median below 0 {x['rare'] < 0}, frequent median above 0 {x['frequent'] > 0}")
print(f"model time {time.time() - t0:.0f} s")

In [ ]:
neural = {g: abs(np.mean([fig6d_vals[m] for m in mice])) for g, mice in (("rare", RARE), ("frequent", FREQ))}
print("neural across-mouse mean magnitude (fig6d, main):", {g: round(v, 3) for g, v in neural.items()})
for (pname, g), runs in model.items():
    pri = prior if pname.startswith("deposit") else prior_img
    R = np.mean([M.reconstruct_prior(r[0]) for r in runs], axis=0)
    R = R / R.sum()
    v = M.dkl(pri["rare"], R) - M.dkl(pri["frequent"], R)
    v_nats = v * np.log(2)
    ok_sign = v < 0 if g == "rare" else v > 0
    print(f"{pname}, trained on {g}: averaged reconstructed prior Delta D_KL {v:.3f} bits ({v_nats:.3f} nats; "
          f"expected sign {ok_sign}; magnitude below neural {abs(v) < neural[g]})")

**Result: fig7 reproduced; fig7g reproduced.** With the code's scale and noise that the answer
fixed, the network learns: the weights end with a mean of 2.25 (rare-trained) and 2.77
(frequent-trained) and reach the cap of 5. The median Delta D_KL of the 50 rare-trained models is
-0.564 (interquartile range -0.614 to -0.529; all 50 below 0) and that of the 50 frequent-trained
models 0.093 (0.031 to 0.200; 40 of 50 above 0), so the criterion holds; the code's likelihood
widths, variants, give the same signs (rare -0.697 and -0.792, frequent 0.774 and 1.003), and so do
the earlier run's priors from the dandiset's trials (-0.529 and 0.101). For fig7g, the averaged
reconstructed prior has Delta D_KL -0.098 bits for the rare-trained and 0.626 bits for the
frequent-trained models (-0.068 and 0.434 nats), both of the expected sign and smaller in
magnitude than the neural across-mouse means of fig6d-prior-reconstruction (0.223 and 1.125); the
rare neural mean has the wrong sign there, and only its magnitude enters this comparison. The
test stimuli, the f_h of S from -0.3 to 1.3 as the code computes it, span 0.04 to 38.45 f_h, not
5.5 to 38.5 as the criterion says, since f_h falls again for S below -0.1.

## edf3-simulated-posterior-population

> A simulated population that stochastically encodes samples from the rare or frequent morph
> posterior reproduces the recorded remapping patterns.

Answer of 2026-10-22: the neurons' centres are evenly spaced over the M6 grid (0.06 to 43.9 f_h),
as the earlier run already had them, and the simulated difference is that of the mean over the
1,000 simulations, with each simulation's trial-by-trial matrix z-scored over its off-diagonal
entries before averaging; the mean z-scored matrices are then averaged within the S-tilde blocks.
The earlier run z-scored each session's 5 x 5 block matrix instead; that is kept as a variant. The
posteriors are those of the deposit's priors.

In [ ]:
FH_GRID = M.unit_to_fh(M.GRID)
SIM_POST = {g: POST[("methods var 8.23", g)] for g in ("rare", "frequent")}
SIM_SESSIONS = []
for s_ in pop:
    rows_ = np.abs(M.GRID[None, :] - s_["trials"].unit.to_numpy()[:, None]).argmin(1)
    cdf_ = np.cumsum(SIM_POST[s_["group"]][rows_], axis=1)
    cdf_ /= cdf_[:, -1:]
    flat_ = (cdf_ + np.arange(len(rows_))[:, None]).ravel()        # rows stacked into one increasing array
    SIM_SESSIONS.append((s_["mouse"], s_["n_cells"], flat_, s_["trials"].morph.to_numpy()))


def _one_thread():
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)


def simulate(args):
    """Sum over simulations of the binned similarity per mouse, for one reading: of the trial
    matrix z-scored over its off-diagonal entries (main), and the earlier run's z-scored 5 x 5
    block matrix (variant).

    Reading a: one sample per trial and the session's cell count; b: 45 samples (one per position
    bin) and 100 neurons. Each trial's samples come from the group posterior at the trial's f_h by
    inverse transform sampling on the grid; neuron activity is gamma(kappa, 1)."""
    reading, n_sims, seed = args
    rng = np.random.default_rng(seed)
    acc, acc_v, cnt = {}, {}, {}
    for _ in range(n_sims):
        per_mouse = {}
        for mouse, ncells, flat, morph in SIM_SESSIONS:
            T = len(morph)
            n_samp, n_neur = (1, ncells) if reading == "a" else (45, 100)
            mu = np.linspace(FH_GRID[0], FH_GRID[-1], n_neur)
            u = rng.random((T, n_samp)) + np.arange(T)[:, None]
            k = np.searchsorted(flat, u) - np.arange(T)[:, None] * len(M.GRID)
            s_fh = FH_GRID[np.clip(k, 0, len(M.GRID) - 1)]
            kappa = 2 * np.exp(-((s_fh[:, :, None] - mu[None, None, :]) ** 2) / 4.11 ** 2)
            y = rng.gamma(np.maximum(kappa, 1e-300), 1.0).reshape(T, -1)
            yn = M.unit_rows(y)
            S = yn @ yn.T
            off = ~np.eye(T, dtype=bool)
            Z = (S - S[off].mean()) / S[off].std()           # trial matrix z-scored over its off-diagonal entries
            per_mouse.setdefault(mouse, []).append((M.binned_similarity(Z, morph),
                                                    M.zscore_matrix(M.binned_similarity(S, morph))))
        for mouse, Bs in per_mouse.items():
            acc[mouse] = acc.get(mouse, 0) + np.mean([b[0] for b in Bs], axis=0)
            acc_v[mouse] = acc_v.get(mouse, 0) + np.mean([b[1] for b in Bs], axis=0)
            cnt[mouse] = cnt.get(mouse, 0) + 1
    return acc, acc_v, cnt


t0 = time.time()
sim = {}
for reading in "ab":
    jobs = [(reading, 250, 9000 + 10 * (reading == "b") + k) for k in range(4)]
    with ProcessPoolExecutor(4, initializer=_one_thread) as ex:
        parts = list(ex.map(simulate, jobs))
    n_done = sum(p[2]["R1"] for p in parts)
    for k, name in ((0, ""), (1, " variant")):
        per = {m: sum(p[k][m] for p in parts) / sum(p[2][m] for p in parts) for m in RARE + FREQ}
        diff = np.mean([per[m] for m in RARE], axis=0) - np.mean([per[m] for m in FREQ], axis=0)
        sim[reading + name] = diff
    print(f"reading {reading}: {n_done} simulations; simulated rare minus frequent, trial matrices z-scored (main):")
    print(pd.DataFrame(sim[reading], index=M.MORPHS, columns=M.MORPHS).round(3).to_string())
    print(f"  variant, block matrices z-scored (earlier run): S-tilde = 0.25 row {sim[reading + ' variant'][1].round(3)}")
obs, perms, sig = fig3f["main: smoothed deconvolved, z-scored per session"]
sig_entries = [j for j in range(5) if sig[1, j]]
print("recorded significant entries of the S-tilde = 0.25 row (fig3f):", [(M.MORPHS[j], round(obs[1, j], 3)) for j in sig_entries])
for reading in ("a", "b", "a variant", "b variant"):
    same = [bool(np.sign(sim[reading][1, j]) == np.sign(obs[1, j])) for j in sig_entries]
    print(f"reading {reading}: values {[round(float(sim[reading][1, j]), 3) for j in sig_entries]}, same sign {same}")
print(f"simulation time {time.time() - t0:.0f} s")

**Result: not reproduced.** The recorded S-tilde = 0.25 row has two significant entries
(fig3f-zscored-difference-at-025), with S-tilde 0.5 (-0.409) and 1.0 (-0.367). With the deposit's
posteriors and each simulation's trial matrix z-scored before averaging, as the answer fixed, the
simulated rare minus frequent difference over 1,000 simulations has the same sign at 0.5 under
both readings (-0.410 with one sample per trial and the session's cell count, -0.444 with 45
samples and 100 neurons). At 1.0 it is near zero: -0.036 under reading (a), the recorded sign, and
+0.030 under reading (b). The criterion asks for both readings, so it is not met. The earlier
run's z-scoring of the block matrices, kept as a variant, gives -0.006 and -0.004 at 1.0 with the
deposit's posteriors (the earlier run had +0.009 under both readings with the dandiset's
priors).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Notebook | Outcome |
|---|---|---|
| cells-per-session | 1 | reproduced |
| session-and-trial-counts | 1 | partially reproduced |
| vr-deposit-link (link) | 2 | reproduced |
| training-intermediate-fraction | 2 | reproduced (data elsewhere) |
| place-cell-fractions | 1 | not reproduced |
| edf4a-fewer-place-cells-frequent | 1 | reproduced |
| edf4a-overlap-above-chance | 1 | partially reproduced |
| edf4bc-overlap-cells-at-ends-and-reward | 1 | not reproduced |
| fig2eh-place-cell-similarity-differs | 1 | partially reproduced |
| edf6-nmf-confirms-remapping | 1 | not reproduced |
| edf5-umap-rare-frequent-separable | 1 | reproduced |
| edf1-licking-before-reward | 1 | reproduced |
| fig3ab-block-versus-gradual-similarity | 2 | not reproduced |
| edf7-checkerboard-sessions | 2 | partially reproduced |
| fig3f-zscored-difference-at-025 | 2 | reproduced |
| fig4-sf-stable-along-track | 2 | reproduced |
| fig6a-dkl-sign | 2 | partially reproduced |
| fig6bc-pooled-dkl | 2 | partially reproduced |
| fig6d-prior-reconstruction | 2 | not reproduced |
| map-estimate-values | 2 | reproduced |
| edf9-early-sessions | 2 | reproduced |
| edf9k-null-models-rare | 2 | reproduced |
| edf9l-null-models-frequent | 2 | partially reproduced |
| edf10-decision-mice-like-frequent | 2 | reproduced |
| fig7-model-reproduces-remapping | 2 | reproduced (data elsewhere) |
| fig7g-model-prior-reconstruction | 2 | reproduced |
| edf3-simulated-posterior-population | 2 | not reproduced |
| edf2-fh-best-predictor | - | blocked: data missing |
| supp-earlier-morph-studies | - | blocked: other |

How the analysis differs from the paper's. The priors of every posterior-based claim come from
the authors' figshare deposit of all VR sessions (answer of 2026-10-22), matched to the dandiset's
files through the 2026 code's mouse aliases and the trial sequences (vr-deposit-link); the
fig6a, fig6bc, fig6d, map-estimate-values, edf9, edf9k, edf9l, edf10, fig7g and edf3 results are
counted as `with_inputs`, and training-intermediate-fraction and fig7 as `elsewhere`. The dandiset
holds 26 rare morph sessions 8 to N, two more than the paper's 24. Where the claims file leaves a
reading open, every reading it names is run (speed exclusion, denominators, likelihood widths,
shuffle counts, decision-mouse sessions, peak readings of edf4bc). Choices not fixed by the claims
file are standard ones, named where they are made: smoothing of empty position bins by normalised
convolution, the 99.9th percentile of the shuffles for the UMAP filter, an NMF iteration limit of
2,000, and the operational test of "rises" (edf1). The Hebbian model follows the parameters and
noise of the paper's code, as the answer fixed; the claims file's statement that its test stimuli
lie between 5.5 and 38.5 f_h does not hold for S below -0.1 (fig7 conclusion).

What was found about the dataset. The version pinned holds all 16 mice; the 2021 versions hold
10. Every file holds only the imaged trials; the non-imaged trials and the sessions without imaging
(2, 4 and 6) are not in the dandiset, though the paper's priors and its training statistics need
them. The authors' figshare deposit holds them, but neither the dandiset nor the paper names it;
its sessions link to the dandiset's files exactly, through the mouse aliases of the 2026 code,
which the dandiset does not record. The deposit also holds sessions with the imaging flag that
have no file in the dandiset (26 for the 12 mice, 12 of them R6's). The files hold imaging sessions 5 and 7 (and R4's session 6), which the paper does not
describe, and F6's session 5 has 4,103 cells, above the paper's range. FD3 has no session after 7,
so the decision-mouse counts cannot match the paper's. `session_start_time` is the date of
conversion (2026), not of the recording. FD1's sessions 1 and 3 hold continuous morph values. The
place cells stored in the JSON annotation (the authors' 2026 code) give fractions like this
attempt's and far above the paper's. No recording is deposited twice, and the activity shares the
clock of the position in every session.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min")